# Hausa Sentiment Analysis
Sentiment classification (positive / neutral / negative) for Hausa-language tweets using the NaijaSenti dataset and AfriBERTa fine-tuning.

## 1. Install & Imports

In [ ]:
!pip install -q datasets transformers accelerate evaluate scikit-learn pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset, DatasetDict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, pipeline

## 2. Load Dataset

In [ ]:
base = "hf://datasets/HausaNLP/NaijaSenti-Twitter@refs/convert/parquet"

ds = DatasetDict({
    "train":      load_dataset("parquet", data_files=f"{base}/hau/train/0000.parquet", split="train"),
    "validation": load_dataset("parquet", data_files=f"{base}/hau/validation/0000.parquet", split="train"),
    "test":       load_dataset("parquet", data_files=f"{base}/hau/test/0000.parquet", split="train"),
})

print(ds)
print("\nLabel names:", ds["train"].features["label"].names)
print("\nSample tweet:", ds["train"][0])

In [ ]:
tr = ds["train"].to_pandas()
va = ds["validation"].to_pandas()
te = ds["test"].to_pandas()

label_names = ds["train"].features["label"].names

print("Train size:", len(tr))
print("Validation size:", len(va))
print("Test size:", len(te))
print("\nLabel distribution (train):")
print(tr["label"].map(dict(enumerate(label_names))).value_counts())

## 3. Baseline — TF-IDF + Logistic Regression

In [ ]:
def run_baseline(max_f, ngram, eval_df):
    vec = TfidfVectorizer(max_features=max_f, ngram_range=ngram)
    clf = LogisticRegression(max_iter=1000).fit(vec.fit_transform(tr["tweet"]), tr["label"])
    p = clf.predict(vec.transform(eval_df["tweet"]))
    return accuracy_score(eval_df["label"], p), f1_score(eval_df["label"], p, average="macro")

rows = []
for mf in [2000, 5000, 20000]:
    for ng in [(1, 1), (1, 2)]:
        a, f = run_baseline(mf, ng, va)
        rows.append(dict(max_features=mf, ngram=str(ng), val_acc=round(a, 4), val_f1=round(f, 4)))

g = pd.DataFrame(rows)
print(g.to_string(index=False))

In [ ]:
best = g.sort_values("val_f1").iloc[-1]
a, f = run_baseline(int(best.max_features), eval(best.ngram), te)
print(f"Best config: max_features={int(best.max_features)}, ngram={best.ngram}")
print(f"TEST accuracy={a:.4f}  macro_f1={f:.4f}")

## 4. Fine-tuning Experiment 1 — AfriBERTa lr=3e-5

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/hausa-sentiment/runs', exist_ok=True)

In [ ]:
def finetune(model_name, lr, epochs, out):
    tok = AutoTokenizer.from_pretrained(model_name)
    names = ds["train"].features["label"].names

    def enc(batch):
        return tok(batch["tweet"], truncation=True, max_length=128)

    encoded = ds.map(enc, batched=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(names),
        id2label=dict(enumerate(names)),
        label2id={n: i for i, n in enumerate(names)},
    )

    def compute_metrics(p):
        preds = np.argmax(p.predictions, axis=1)
        return {
            "accuracy": accuracy_score(p.label_ids, preds),
            "macro_f1": f1_score(p.label_ids, preds, average="macro"),
        }

    args = TrainingArguments(
        output_dir=out,
        learning_rate=lr,
        num_train_epochs=epochs,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        weight_decay=0.01,
        seed=42,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=encoded["train"],
        eval_dataset=encoded["validation"],
        processing_class=tok,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    results = trainer.evaluate(encoded["test"])
    print("TEST:", results)
    trainer.save_model(out)
    tok.save_pretrained(out)
    return results

In [ ]:
exp1 = finetune(
    "castorini/afriberta_base",
    lr=3e-5,
    epochs=5,
    out="/content/drive/MyDrive/hausa-sentiment/runs/afriberta_3e-5"
)

## 5. Fine-tuning Experiment 2 — AfriBERTa lr=1e-5

In [ ]:
exp2 = finetune(
    "castorini/afriberta_base",
    lr=1e-5,
    epochs=5,
    out="/content/drive/MyDrive/hausa-sentiment/runs/afriberta_1e-5"
)

## 6. Fine-tuning Experiment 3 — XLM-R base lr=2e-5

In [ ]:
exp3 = finetune(
    "xlm-roberta-base",
    lr=2e-5,
    epochs=5,
    out="/content/drive/MyDrive/hausa-sentiment/runs/xlmr_2e-5"
)

## 7. Results Summary

In [ ]:
results_table = pd.DataFrame([
    {"Experiment": "Baseline",         "Model": "TF-IDF + Logistic Regression", "LR": "-",   "Test Acc": 0.7247, "Test Macro-F1": 0.7243},
    {"Experiment": "Exp 1",            "Model": "AfriBERTa-base",               "LR": "3e-5", "Test Acc": round(exp1["eval_accuracy"], 4), "Test Macro-F1": round(exp1["eval_macro_f1"], 4)},
    {"Experiment": "Exp 2",            "Model": "AfriBERTa-base",               "LR": "1e-5", "Test Acc": round(exp2["eval_accuracy"], 4), "Test Macro-F1": round(exp2["eval_macro_f1"], 4)},
    {"Experiment": "Exp 3",            "Model": "XLM-R base",                   "LR": "2e-5", "Test Acc": round(exp3["eval_accuracy"], 4), "Test Macro-F1": round(exp3["eval_macro_f1"], 4)},
])
print(results_table.to_string(index=False))

## 8. Evaluation & Error Analysis

In [ ]:
clf = pipeline(
    "text-classification",
    model="/content/drive/MyDrive/hausa-sentiment/runs/afriberta_3e-5",
    truncation=True,
    device=0
)

te["pred_label"] = [clf(t)[0]["label"] for t in te["tweet"]]
te["true_label"] = te["label"].map(clf.model.config.id2label)

print(classification_report(te["true_label"], te["pred_label"]))

In [ ]:
labels = list(clf.model.config.id2label.values())
cm = confusion_matrix(te["true_label"], te["pred_label"], labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print(cm_df)

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels)
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, cm[i, j], ha="center", va="center")
plt.tight_layout()
plt.savefig("/content/hausa-sentiment/confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
errors = te[te["true_label"] != te["pred_label"]]
correct = te[te["true_label"] == te["pred_label"]]

errors.sample(min(40, len(errors)), random_state=1).to_csv("/content/hausa-sentiment/errors_sample.csv", index=False)
correct.sample(min(20, len(correct)), random_state=1).to_csv("/content/hausa-sentiment/correct_sample.csv", index=False)

print("Sample errors:")
print(errors[["tweet", "true_label", "pred_label"]].sample(10, random_state=1).to_string(index=False))

In [ ]:
import os
os.chdir('/content/hausa-sentiment')
!git add errors_sample.csv correct_sample.csv confusion_matrix.png
!git commit -m "added evaluation results and error analysis"
!git push origin main